In [ ]:
# Prerequisites (Notebook → Settings, right sidebar):
#   1. Internet = ON  (required for git clone + pip)
#   2. Accelerator = GPU (T4 recommended)
#   3. Input dataset = LLVIP (wait until 100% downloaded before running)

import os
REPO = '/kaggle/working/microghost'
BRANCH = 'Ibtida'  # use 'main' for default branch

if os.path.isdir(os.path.join(REPO, '.git')):
    print('Repo already cloned — pulling latest...')
    %cd {REPO}
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} https://github.com/Sayjad21/microghost.git {REPO}
    %cd {REPO}

In [ ]:
%cd /kaggle/working/microghost
!pip install -r requirements.txt -q
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q
print('Dependencies installed.')

In [ ]:
import os

def find_llvip_root(base='/kaggle/input'):
    """Find folder containing infrared/ + Annotations/ (LLVIP layout)."""
    if not os.path.isdir(base):
        return None
    # List top-level mounts first (fast)
    for name in os.listdir(base):
        root = os.path.join(base, name)
        if not os.path.isdir(root):
            continue
        # Dataset may be nested: /kaggle/input/llvip-dataset/LLVIP/...
        candidates = [root]
        for sub in os.listdir(root):
            p = os.path.join(root, sub)
            if os.path.isdir(p):
                candidates.append(p)
        for cand in candidates:
            dirs = {d.lower() for d in os.listdir(cand) if os.path.isdir(os.path.join(cand, d))}
            if 'infrared' in dirs and 'annotations' in dirs:
                return cand
    return None

base_input = '/kaggle/input'
llvip_path = find_llvip_root(base_input)

print('Mounted under /kaggle/input:')
if os.path.isdir(base_input):
    for item in os.listdir(base_input):
        print(f'  - {item}')
else:
    print('  (empty — add LLVIP dataset in sidebar and wait for 100% download)')

if llvip_path:
    print(f'\n✅ LLVIP root: {llvip_path}')
    n_ir = len(os.listdir(os.path.join(llvip_path, 'infrared', 'train')))
    print(f'   infrared/train samples: {n_ir}')
else:
    print('\n❌ LLVIP not found.')
    print('   Add dataset via sidebar → Input → Add → search LLVIP')
    print('   Wait until download bar shows 100%, then re-run this cell.')

In [ ]:
import os

# Auto-detect the LLVIP dataset path inside Kaggle
base_input = '/kaggle/input'
llvip_path = '/datasets/saisumathappala/llvip-dataset'

if os.path.exists(base_input):
    for root, dirs, files in os.walk(base_input):
        lower_dirs = [d.lower() for d in dirs]
        if 'infrared' in lower_dirs and 'annotations' in lower_dirs:
            llvip_path = root
            break

if llvip_path:
    print(f"✅ Found LLVIP dataset at: {llvip_path}")
    # Run your main.py training script with auto-detected path
    # Scale up batch-size and num-workers for Kaggle GPUs (T4/P100)
    !python main.py train --dataset llvip --data-root {llvip_path} --batch-size 32 --num-workers 2
    
    print("\n🚀 Training Complete! Running Visual Diagnostics...")
    !python compare_models.py --dataset llvip --data-root {llvip_path}
else:
    print("❌ Could not find the LLVIP dataset in /kaggle/input.")
    print("Here is what is currently mounted in /kaggle/input:")
    if os.path.exists(base_input):
        for item in os.listdir(base_input):
            print(f" - {item}")
    print("\nPlease ensure you have added the dataset to the notebook using the right sidebar! If the folder name is listed above, you can manually set the path.")

In [ ]:
# Optional: compare v1 vs v2 (requires v1 checkpoint in checkpoints/)
import os
%cd /kaggle/working/microghost
v1 = 'checkpoints/best_microghost_thermal_v1.pth'
if os.path.isfile(v1):
    !python compare_models.py --dataset llvip --data-root {llvip_path}
else:
    print('Skipping compare_models — upload v1 checkpoint or train v1 first.')
    print('You can still run inference with the newly trained v2 checkpoint.')